# Dashboard final: siniestros viales

Este notebook integra los resultados del flujo de trabajo del TP para comunicar el problema, los hallazgos principales y el desempeno del modelo seleccionado. Consume el dataset procesado y los artefactos ya guardados en `outputs/`; no modifica el EDA, los modelos, el entrenamiento ni las metricas.


## Contexto del problema

El proyecto analiza registros de siniestros viales con el objetivo de comprender patrones de gravedad y evaluar si las caracteristicas de la victima y del contexto permiten anticipar casos graves o mortales. La comunicacion final busca combinar lectura descriptiva del fenomeno con evidencia del modelo predictivo.

## Hipotesis

La hipotesis de modelado plantea que las caracteristicas de la victima y del contexto del siniestro permiten anticipar si el caso terminara siendo grave o mortal. Para evitar leakage, el modelo excluye variables directamente asociadas al target o a la gravedad observada.

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "siniestros_limpio_enriquecido.csv"
MODEL_METRICS_FILE = PROJECT_ROOT / "outputs" / "model_metrics.json"
MODEL_COMPARISON_FILE = PROJECT_ROOT / "outputs" / "model_comparison.json"
CROSS_VALIDATION_FILE = PROJECT_ROOT / "outputs" / "cross_validation_results.json"
DASHBOARD_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "dashboards"
FIGURES_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures"
DASHBOARD_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
FIGURES_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
dashboard_path = DASHBOARD_OUTPUT_DIR / "dashboard_siniestros.html"
confusion_matrix_path = FIGURES_OUTPUT_DIR / "confusion_matrix_random_forest.png"
LOG_FILE = PROJECT_ROOT / "logs" / "pipeline.log"

LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger('dashboard_storytelling')
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8-sig')
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s'))
logger.addHandler(handler)
logger.info('Inicio de generacion del dashboard final de siniestros viales')

with MODEL_METRICS_FILE.open(encoding='utf-8') as file:
    model_metrics = json.load(file)
with MODEL_COMPARISON_FILE.open(encoding='utf-8') as file:
    model_comparison = json.load(file)
with CROSS_VALIDATION_FILE.open(encoding='utf-8') as file:
    cross_validation = json.load(file)

df = pd.read_csv(DATA_FILE)
df['fecha_siniestro'] = pd.to_datetime(df['fecha_siniestro'], errors='coerce')
df['gravedad_victima'] = df['gravedad_victima'].fillna('SIN_DATO')
df['edad_grupo'] = df['edad_grupo'].fillna('SIN_DATO')
df['vulnerabilidad_usuario'] = df['vulnerabilidad_usuario'].fillna('SIN_DATO')

gravedad_order = ['LEVE', 'GRAVE', 'MORTAL', 'SIN_DATO']
edad_order = ['menor_18', '18_30', '31_45', '46_60', 'mayor_60', 'SIN_DATO']

selected_model = cross_validation.get('best_model', model_comparison.get('best_model'))
best_f1 = cross_validation.get('best_model_metrics', {}).get('f1_mean', model_comparison.get('best_model_metrics', {}).get('f1'))
features = cross_validation.get('features', {})
feature_count = len(features.get('numeric', [])) + len(features.get('categorical', []))
severe_rate = df['es_grave_o_mortal'].mean() * 100

kpis = {
    'Registros totales': f'{len(df):,}'.replace(',', '.'),
    'Casos graves o mortales': f'{severe_rate:.2f}%',
    'Modelo seleccionado': selected_model,
    'Mejor F1 Score CV': f'{best_f1:.3f}',
    'Features usadas': str(feature_count),
}

severity_counts = (
    df['gravedad_victima']
    .value_counts()
    .reindex([value for value in gravedad_order if value in df['gravedad_victima'].unique()])
    .fillna(0)
    .astype(int)
)
severity_percentages = severity_counts / severity_counts.sum() * 100
leve_pct = float(severity_percentages.get('LEVE', 0.0))
grave_pct = float(severity_percentages.get('GRAVE', 0.0))
mortal_pct = float(severity_percentages.get('MORTAL', 0.0))
leve_count = int(severity_counts.get('LEVE', 0))
grave_count = int(severity_counts.get('GRAVE', 0))
mortal_count = int(severity_counts.get('MORTAL', 0))
leve_count_label = f'{leve_count:,}'.replace(',', '.')
grave_count_label = f'{grave_count:,}'.replace(',', '.')
mortal_count_label = f'{mortal_count:,}'.replace(',', '.')

winner_payload = model_comparison.get('models', {}).get(selected_model, {})
rf_metrics = winner_payload.get('metrics', model_comparison.get('best_model_metrics', {}))
rf_accuracy = float(rf_metrics.get('accuracy', 0.0))
rf_precision = float(rf_metrics.get('precision', 0.0))
rf_recall = float(rf_metrics.get('recall', 0.0))
rf_f1 = float(rf_metrics.get('f1', 0.0))

cm = winner_payload.get('confusion_matrix')
if not cm:
    raise ValueError(f'No se encontro confusion_matrix para {selected_model} en outputs/model_comparison.json')
tn, fp = cm[0]
fn, tp = cm[1]

cv_best_metrics = cross_validation.get('best_model_metrics', {})
best_f1_std = float(cv_best_metrics.get('f1_std', 0.0))

model_summary_df = pd.DataFrame(model_comparison.get('comparison_table', []))
model_summary_df = model_summary_df[['modelo', 'accuracy', 'precision', 'recall', 'f1']].copy()
model_summary_df['ganador'] = model_summary_df['modelo'].eq(selected_model)
model_summary_df = model_summary_df.rename(
    columns={
        'modelo': 'Modelo',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1': 'F1',
    }
)

imbalance_explanation = (
    'Las victimas leves representan aproximadamente el 95% de los registros. Los casos graves y mortales '
    'constituyen una proporcion muy reducida del total, evidenciando un fuerte desbalanceo de clases que '
    'condiciona el entrenamiento y la evaluacion de los modelos predictivos.'
)
imbalance_conclusion = (
    'Debido a este desbalanceo se priorizaron Recall y F1 Score por sobre Accuracy para seleccionar el modelo final.'
)
model_selection_conclusion = (
    'RandomForestClassifier obtuvo el mejor equilibrio entre capacidad predictiva, estabilidad y generalizacion.'
)
confusion_interpretation = (
    f'El modelo detecto correctamente {tp} casos graves o mortales y solo dejo sin detectar {fn} casos. '
    'Esto indica una alta capacidad para identificar situaciones de riesgo.'
)
false_positive_text = 'Falso Positivo: se clasifica como grave un caso que no lo era.'
false_negative_text = 'Falso Negativo: se clasifica como leve un caso realmente grave.'
error_cost_conclusion = (
    'En este problema resulta mas costoso un falso negativo, ya que implica no detectar una situacion '
    'potencialmente critica.'
)
interpretacion_desbalanceo = f'{imbalance_explanation} {imbalance_conclusion}'
interpretacion_distribucion = (
    'La distribucion confirma que la mayoria de los siniestros registrados no derivan en lesiones severas. '
    'Sin embargo, la baja frecuencia de eventos criticos no reduce su relevancia, ya que representan los casos '
    'de mayor impacto social y sanitario.'
)
interpretacion_edad = (
    'Los distintos grupos etarios presentan una composicion relativamente similar, con predominio de casos leves. '
    'No obstante, algunos segmentos muestran una mayor proporcion relativa de lesiones graves o mortales, sugiriendo '
    'posibles diferencias de vulnerabilidad segun la edad.'
)
interpretacion_vulnerabilidad = (
    'La vulnerabilidad del usuario parece asociarse con cambios en la proporcion de casos graves y mortales. '
    'Los grupos catalogados con mayor vulnerabilidad concentran una participacion relativamente superior de eventos severos.'
)
interpretacion_temporal = (
    'La serie temporal muestra variaciones importantes durante el periodo analizado. Las fluctuaciones observadas '
    'podrian relacionarse con cambios en la movilidad urbana y otros factores contextuales.'
)
interpretacion_modelos = (
    'Los modelos presentan comportamientos distintos frente al problema de clasificacion. RandomForestClassifier '
    'logra el mejor equilibrio general entre Accuracy, Precision, Recall y F1.'
)
interpretacion_cv = (
    'RandomForestClassifier obtuvo el mayor F1 promedio y mantuvo una baja variabilidad entre folds, evidenciando '
    'una mejor capacidad de generalizacion.'
)
interpretacion_matriz = (
    'El modelo logra identificar correctamente la mayoria de los casos graves o mortales, manteniendo una cantidad '
    'reducida de falsos negativos.'
)
if rf_recall > rf_precision:
    confusion_conclusion = (
        'El Recall es mayor que la Precision: el modelo prioriza detectar casos graves o mortales, '
        'aunque eso implique generar mas falsos positivos. Esta conducta es razonable cuando el costo de no detectar '
        'un caso grave/mortal es mas alto que el costo de revisar alertas adicionales.'
    )
else:
    confusion_conclusion = (
        'La Precision es mayor o igual que el Recall: el modelo es mas conservador al marcar casos graves o mortales, '
        'pero puede dejar una proporcion mayor de positivos reales sin detectar.'
    )
formatted_total_records = f'{len(df):,}'.replace(',', '.')
final_conclusion = (
    f'El proyecto analizo {formatted_total_records} registros y encontro que los casos graves o mortales representan '
    f'{severe_rate:.2f}% del dataset. Los resultados muestran que es posible anticipar parcialmente la gravedad '
    'de un siniestro utilizando informacion disponible sobre la victima y el contexto del hecho. '
    f'El modelo {selected_model} obtuvo el mejor equilibrio entre deteccion de casos criticos y estabilidad de '
    f'generalizacion, con F1 promedio de {best_f1:.3f} y Recall en test de {rf_recall:.3f}.'
)
limitations = [
    'Fuerte desbalanceo del dataset.',
    'Ausencia de variables contextuales adicionales.',
    'Calidad dependiente de registros administrativos.',
    'Simplificacion del target en clasificacion binaria.',
]
future_work = [
    'Incorporacion de nuevas variables contextuales.',
    'Analisis geografico.',
    'Validacion temporal.',
    'Calibracion de probabilidades.',
    'Evaluacion de tecnicas especificas para datasets desbalanceados.',
]

def _nbformat_disponible():
    try:
        import nbformat
        version = tuple(int(part) for part in nbformat.__version__.split('.')[:2])
        return version >= (4, 2)
    except Exception:
        return False

def _mostrar_mensaje_render(mensaje):
    try:
        from IPython.display import HTML, display
        display(HTML(f"<p style='color:#6c757d'>{mensaje}</p>"))
    except Exception:
        print(mensaje)

PLOTLY_LIGHT_THEME = {
    "template": "plotly_white",
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": dict(
        family="Arial",
        size=14,
        color="black",
    ),
    "title_font": dict(
        size=20,
        color="black",
    ),
    "legend": dict(
        bgcolor="white",
        bordercolor="lightgray",
        borderwidth=1,
        font=dict(color="black"),
    ),
}

AXIS_LIGHT_THEME = dict(
    showgrid=True,
    gridcolor="lightgray",
    zerolinecolor="lightgray",
    color="black",
    title_font=dict(color="black"),
    tickfont=dict(color="black"),
)

def apply_plotly_light_theme(figure, height=520):
    figure.update_layout(**PLOTLY_LIGHT_THEME)
    if figure.layout.height is None:
        figure.update_layout(height=height)
    figure.update_xaxes(**AXIS_LIGHT_THEME)
    figure.update_yaxes(**AXIS_LIGHT_THEME)
    for annotation in figure.layout.annotations:
        annotation.font = dict(color="black")
    return figure

def show_fig(figure):
    if 'ipykernel' not in sys.modules:
        return

    if _nbformat_disponible():
        try:
            figure.show()
            return
        except Exception as exc:
            render_error = exc
    else:
        render_error = RuntimeError('nbformat>=4.2.0 no esta disponible para el render MIME de Plotly')

    try:
        fallback_path = DASHBOARD_OUTPUT_DIR / f"plotly_render_fallback_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.html"
        figure.write_html(fallback_path, include_plotlyjs='cdn', full_html=True)
        mensaje = f"No se pudo renderizar la figura en el notebook. Se exporto un HTML temporal en: {fallback_path}"
        logger.warning('%s Error original: %s', mensaje, render_error)
        _mostrar_mensaje_render(mensaje)
    except Exception as html_error:
        mensaje = f"No se pudo renderizar ni exportar la figura de Plotly. Error original: {render_error}. Error HTML: {html_error}"
        logger.warning(mensaje)
        _mostrar_mensaje_render(mensaje)

logger.info('Datos y artefactos cargados para dashboard: filas=%s, modelo=%s', len(df), selected_model)
print(kpis)


## KPIs principales

Los KPIs resumen la escala del dataset, la proporcion de casos graves o mortales, el modelo elegido, su mejor F1 promedio de validacion cruzada y la cantidad de variables predictoras usadas.

In [ ]:
kpi_fig = make_subplots(rows=1, cols=5, specs=[[{'type': 'indicator'} for _ in range(5)]])
for index, (title, value) in enumerate(kpis.items(), start=1):
    mode = 'number' if title in {'Registros totales', 'Features usadas'} else 'number'
    display_value = value
    if title == 'Registros totales':
        numeric_value = len(df)
        suffix = ''
    elif title == 'Features usadas':
        numeric_value = feature_count
        suffix = ''
    elif title == 'Casos graves o mortales':
        numeric_value = severe_rate
        suffix = '%'
    elif title == 'Mejor F1 Score CV':
        numeric_value = best_f1
        suffix = ''
    else:
        numeric_value = 0
        suffix = ''
    if title == 'Modelo seleccionado':
        kpi_fig.add_trace(go.Indicator(mode='number', value=0, title={'text': f'<b>{title}</b><br><span style="font-size:20px">{display_value}</span>'}, number={'font': {'size': 1}, 'valueformat': ' '}), row=1, col=index)
    else:
        kpi_fig.add_trace(go.Indicator(mode=mode, value=numeric_value, title={'text': f'<b>{title}</b>'}, number={'suffix': suffix}), row=1, col=index)

kpi_fig.update_layout(height=220, margin=dict(l=20, r=20, t=30, b=20))
apply_plotly_light_theme(kpi_fig)
show_fig(kpi_fig)

## Desbalanceo del dataset

Se resume la distribucion de severidad observada para justificar por que Accuracy no alcanza como metrica principal en este problema.


In [ ]:
severity_distribution_df = severity_counts.rename_axis("gravedad_victima").reset_index(name="cantidad")
severity_distribution_df["porcentaje"] = severity_distribution_df["cantidad"] / severity_distribution_df["cantidad"].sum() * 100

fig_desbalanceo = px.bar(
    severity_distribution_df,
    x="gravedad_victima",
    y="porcentaje",
    text=severity_distribution_df["porcentaje"].map(lambda value: f"{value:.1f}%"),
    color="gravedad_victima",
    color_discrete_map={"LEVE": "#2E86AB", "GRAVE": "#F18F01", "MORTAL": "#C73E1D", "SIN_DATO": "#6C757D"},
    title="Desbalanceo del dataset por gravedad",
    labels={"gravedad_victima": "Gravedad", "porcentaje": "Porcentaje del total"},
)
fig_desbalanceo.update_layout(showlegend=False, yaxis_ticksuffix="%")
fig_desbalanceo.update_traces(textposition="outside")
apply_plotly_light_theme(fig_desbalanceo)
show_fig(fig_desbalanceo)

print(f"LEVE: {leve_pct:.2f}% | GRAVE: {grave_pct:.2f}% | MORTAL: {mortal_pct:.2f}%")
print(imbalance_explanation)
print(imbalance_conclusion)


## Visualizaciones descriptivas y predictivas

Las visualizaciones combinan lectura del dataset procesado con resultados del modelo: distribucion de gravedad, perfiles de mayor riesgo, evolucion temporal, comparacion entre modelos y estabilidad por Cross Validation.

In [ ]:
gravedad_counts = (
    df['gravedad_victima']
    .value_counts()
    .reindex([value for value in gravedad_order if value in df['gravedad_victima'].unique()])
    .reset_index()
)
gravedad_counts.columns = ['gravedad_victima', 'cantidad']
gravedad_counts['porcentaje'] = gravedad_counts['cantidad'] / gravedad_counts['cantidad'].sum() * 100

fig_gravedad = px.bar(
    gravedad_counts,
    x='gravedad_victima',
    y='cantidad',
    text=gravedad_counts['porcentaje'].map(lambda value: f'{value:.1f}%'),
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Distribucion de gravedad de las victimas',
)
fig_gravedad.update_layout(showlegend=False, xaxis_title='Gravedad', yaxis_title='Cantidad de registros')
fig_gravedad.update_traces(textposition='outside')
apply_plotly_light_theme(fig_gravedad)
show_fig(fig_gravedad)

edad_gravedad = (
    df.groupby(['edad_grupo', 'gravedad_victima'])
    .size()
    .reset_index(name='cantidad')
)
edad_gravedad['total_grupo'] = edad_gravedad.groupby('edad_grupo')['cantidad'].transform('sum')
edad_gravedad['porcentaje'] = edad_gravedad['cantidad'] / edad_gravedad['total_grupo'] * 100
edad_gravedad['edad_grupo'] = pd.Categorical(edad_gravedad['edad_grupo'], categories=edad_order, ordered=True)
edad_gravedad = edad_gravedad.sort_values('edad_grupo')

fig_edad = px.bar(
    edad_gravedad,
    x='edad_grupo',
    y='porcentaje',
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Composicion de gravedad por grupo etario',
    labels={'porcentaje': 'Porcentaje dentro del grupo', 'edad_grupo': 'Grupo etario'},
)
fig_edad.update_layout(barmode='stack', yaxis_ticksuffix='%')
apply_plotly_light_theme(fig_edad)
show_fig(fig_edad)

vulnerabilidad_gravedad = (
    df.groupby(['vulnerabilidad_usuario', 'gravedad_victima'])
    .size()
    .reset_index(name='cantidad')
)
vulnerabilidad_gravedad['total_grupo'] = vulnerabilidad_gravedad.groupby('vulnerabilidad_usuario')['cantidad'].transform('sum')
vulnerabilidad_gravedad['porcentaje'] = vulnerabilidad_gravedad['cantidad'] / vulnerabilidad_gravedad['total_grupo'] * 100

fig_vulnerabilidad = px.bar(
    vulnerabilidad_gravedad,
    x='vulnerabilidad_usuario',
    y='porcentaje',
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Composicion de gravedad por vulnerabilidad del usuario',
    labels={'porcentaje': 'Porcentaje dentro del grupo', 'vulnerabilidad_usuario': 'Vulnerabilidad'},
)
fig_vulnerabilidad.update_layout(barmode='stack', yaxis_ticksuffix='%')
apply_plotly_light_theme(fig_vulnerabilidad)
show_fig(fig_vulnerabilidad)

temporal = (
    df.dropna(subset=['fecha_siniestro'])
    .assign(periodo=lambda data: data['fecha_siniestro'].dt.to_period('M').dt.to_timestamp())
    .groupby(['periodo', 'es_grave_o_mortal'])
    .size()
    .reset_index(name='cantidad')
)
temporal['resultado'] = temporal['es_grave_o_mortal'].map({0: 'Leve', 1: 'Grave o mortal'})
fig_temporal = px.line(
    temporal,
    x='periodo',
    y='cantidad',
    color='resultado',
    markers=True,
    title='Evolucion temporal mensual de siniestros',
    labels={'periodo': 'Mes', 'cantidad': 'Cantidad de registros', 'resultado': 'Resultado'},
    color_discrete_map={'Leve': '#2E86AB', 'Grave o mortal': '#C73E1D'},
)
apply_plotly_light_theme(fig_temporal)
show_fig(fig_temporal)

comparison = pd.DataFrame(model_comparison.get('comparison_table', []))
comparison_long = comparison.melt(id_vars='modelo', value_vars=['accuracy', 'precision', 'recall', 'f1'], var_name='metrica', value_name='valor')
fig_modelos = px.bar(
    comparison_long,
    x='modelo',
    y='valor',
    color='metrica',
    barmode='group',
    title='Comparacion de metricas en holdout por modelo',
    labels={'modelo': 'Modelo', 'valor': 'Valor', 'metrica': 'Metrica'},
    color_discrete_sequence=['#2E86AB', '#6A4C93', '#F18F01', '#C73E1D'],
)
fig_modelos.update_layout(yaxis_range=[0, 1])
apply_plotly_light_theme(fig_modelos)
show_fig(fig_modelos)

cv_table = pd.DataFrame(cross_validation.get('comparison_table', []))
fig_cv = px.bar(
    cv_table,
    x='modelo',
    y='f1_mean',
    error_y='f1_std',
    color='modelo',
    title='F1 promedio en Cross Validation con desvio estandar',
    labels={'modelo': 'Modelo', 'f1_mean': 'F1 promedio', 'f1_std': 'Desvio estandar'},
    color_discrete_sequence=['#2E86AB', '#F18F01', '#C73E1D'],
)
fig_cv.update_layout(showlegend=False, yaxis_range=[0, max(0.35, cv_table['f1_mean'].max() + 0.05)])
apply_plotly_light_theme(fig_cv)
show_fig(fig_cv)

## Hallazgos principales

- La gran mayoria de los registros corresponde a casos leves, por lo que el problema predictivo esta desbalanceado.
- La proporcion de casos graves o mortales es baja, pero analiticamente relevante porque representa el evento de mayor impacto.
- Las variables de edad, modo de desplazamiento, rol y vulnerabilidad permiten construir una lectura de riesgo sin usar variables que filtran directamente la gravedad observada.
- La comparacion de modelos muestra que el desempeno no debe leerse solo por accuracy: en un dataset desbalanceado, recall y F1 son mas informativos para detectar casos graves o mortales.

## Resultado del modelo

El modelo seleccionado es el de mejor desempeno segun la estrategia de validacion definida. La decision prioriza F1 promedio y estabilidad entre folds, buscando un equilibrio entre deteccion de casos positivos y control de falsos positivos. El resultado es adecuado para un trabajo exploratorio y de trazabilidad academica, pero no debe interpretarse como un sistema listo para decision operativa sin nuevas validaciones.

## Evaluacion detallada del modelo ganador

La matriz de confusion se recupera de los artefactos ya generados por el pipeline de comparacion de modelos. En este problema la clase positiva representa casos graves o mortales, por lo que interesa distinguir cuantos casos positivos fueron detectados y cuantos quedaron sin identificar.

### Interpretacion de la matriz de confusion

- **Verdaderos Negativos (TN):** casos no graves/mortales correctamente clasificados como no graves/mortales.
- **Falsos Positivos (FP):** casos no graves/mortales clasificados por el modelo como graves/mortales.
- **Falsos Negativos (FN):** casos graves/mortales clasificados por el modelo como no graves/mortales.
- **Verdaderos Positivos (TP):** casos graves/mortales correctamente detectados.


In [ ]:
confusion_labels = [[f"Verdaderos Negativos\n{tn}", f"Falsos Positivos\n{fp}"], [f"Falsos Negativos\n{fn}", f"Verdaderos Positivos\n{tp}"]]
plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    cm,
    annot=confusion_labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["Predicho no grave/mortal", "Predicho grave/mortal"],
    yticklabels=["Real no grave/mortal", "Real grave/mortal"],
    linewidths=0.5,
    linecolor="white",
)
ax.set_title("Matriz de confusion - RandomForestClassifier")
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
plt.tight_layout()
plt.savefig(confusion_matrix_path, dpi=160, bbox_inches="tight")
plt.show()
logger.info(f"Matriz de confusion exportada en: {confusion_matrix_path}")

display(model_summary_df)
print(confusion_interpretation)
print(false_positive_text)
print(false_negative_text)
print(error_cost_conclusion)
print(confusion_conclusion)
logger.info(
    "Evaluacion RandomForest desde artefactos: accuracy=%.4f precision=%.4f recall=%.4f f1=%.4f tn=%s fp=%s fn=%s tp=%s",
    rf_accuracy,
    rf_precision,
    rf_recall,
    rf_f1,
    tn,
    fp,
    fn,
    tp,
)


## Limitaciones

- Fuerte desbalanceo del dataset.
- Ausencia de variables contextuales adicionales.
- Calidad dependiente del registro administrativo.
- Simplificacion del target en una clasificacion binaria.


## Trabajos futuros

- Incorporacion de nuevas variables contextuales.
- Analisis geografico.
- Validacion temporal.
- Calibracion de probabilidades.
- Evaluacion de tecnicas para datasets desbalanceados.


In [ ]:
confusion_matrix_relative_path = '../figures/confusion_matrix_random_forest.png'

def add_interpretation(text):
    return f"""
    <aside class='interpretation-block'>
        <h3>Interpretacion</h3>
        <p>{text}</p>
    </aside>
    """

kpi_cards = ''.join(
    f"""
    <article class='kpi-card'>
        <div class='kpi-label'>{label}</div>
        <div class='kpi-value'>{value}</div>
    </article>
    """
    for label, value in kpis.items()
)

plot_sections = [
    ('KPIs principales', kpi_fig, 'Los indicadores principales sintetizan el volumen analizado, la proporcion de casos criticos, el modelo seleccionado y el desempeno validado.'),
    ('Desbalanceo del dataset', fig_desbalanceo, interpretacion_desbalanceo),
    ('Distribucion de gravedad de las victimas', fig_gravedad, interpretacion_distribucion),
    ('Composicion de gravedad por grupo etario', fig_edad, interpretacion_edad),
    ('Composicion de gravedad por vulnerabilidad del usuario', fig_vulnerabilidad, interpretacion_vulnerabilidad),
    ('Evolucion temporal mensual de siniestros', fig_temporal, interpretacion_temporal),
    ('Comparacion de metricas por modelo', fig_modelos, interpretacion_modelos),
    ('Cross Validation', fig_cv, interpretacion_cv),
]

plot_html = ''
for index, (title, figure, interpretation) in enumerate(plot_sections):
    plot_html += f"<section class='chart-section'><h2>{title}</h2>"
    plot_html += pio.to_html(figure, include_plotlyjs='cdn' if index == 0 else False, full_html=False)
    plot_html += add_interpretation(interpretation)
    plot_html += '</section>'

model_rows = ''.join(
    f"""
    <tr class='{ 'winner-row' if row['ganador'] else '' }'>
        <td>{row['Modelo']}</td>
        <td>{row['Accuracy']:.3f}</td>
        <td>{row['Precision']:.3f}</td>
        <td>{row['Recall']:.3f}</td>
        <td>{row['F1']:.3f}</td>
    </tr>
    """
    for _, row in model_summary_df.iterrows()
)

model_table_html = f"""
<section>
    <h2>Tabla resumen de modelos</h2>
    <table class='metrics-table model-summary'>
        <thead><tr><th>Modelo</th><th>Accuracy</th><th>Precision</th><th>Recall</th><th>F1</th></tr></thead>
        <tbody>{model_rows}</tbody>
    </table>
    <p>{model_selection_conclusion}</p>
</section>
"""

imbalance_html = f"""
<section>
    <h2>Desbalanceo del dataset</h2>
    <p><strong>LEVE:</strong> {leve_count_label} ({leve_pct:.2f}%) &nbsp; <strong>GRAVE:</strong> {grave_count_label} ({grave_pct:.2f}%) &nbsp; <strong>MORTAL:</strong> {mortal_count_label} ({mortal_pct:.2f}%)</p>
    <p>{imbalance_explanation}</p>
    <p>{imbalance_conclusion}</p>
</section>
"""

confusion_matrix_html = f"""
<section>
    <h2>Matriz de confusion del modelo ganador</h2>
    <p>La matriz resume los aciertos y errores del RandomForestClassifier sobre el conjunto de test usado en la comparacion de modelos.</p>
    <img class='confusion-matrix' src='{confusion_matrix_relative_path}' alt='Matriz de confusion del RandomForestClassifier'>
    {add_interpretation(interpretacion_matriz)}
    <table class='metrics-table'>
        <thead><tr><th>Accuracy</th><th>Precision</th><th>Recall</th><th>F1</th></tr></thead>
        <tbody><tr><td>{rf_accuracy:.3f}</td><td>{rf_precision:.3f}</td><td>{rf_recall:.3f}</td><td>{rf_f1:.3f}</td></tr></tbody>
    </table>
</section>
"""

confusion_interpretation_html = f"""
<section>
    <h2>Interpretacion</h2>
    <p>{confusion_interpretation}</p>
    <p><strong>Verdaderos Positivos:</strong> {tp} &nbsp; <strong>Verdaderos Negativos:</strong> {tn} &nbsp; <strong>Falsos Positivos:</strong> {fp} &nbsp; <strong>Falsos Negativos:</strong> {fn}</p>
    <p>{confusion_conclusion}</p>
</section>
"""

error_cost_html = f"""
<section>
    <h2>Costo de los errores</h2>
    <p><strong>Falso Positivo:</strong> se clasifica como grave un caso que no lo era.</p>
    <p><strong>Falso Negativo:</strong> se clasifica como leve un caso realmente grave.</p>
    <p>{error_cost_conclusion}</p>
</section>
"""

conclusions_html = f"""
<section>
    <h2>Conclusiones</h2>
    <p><strong>Registros analizados:</strong> {formatted_total_records}</p>
    <p><strong>Casos graves o mortales:</strong> {severe_rate:.2f}%</p>
    <p><strong>Modelo ganador:</strong> {selected_model}</p>
    <p><strong>Mejor F1 Score:</strong> {best_f1:.3f} &nbsp; <strong>Recall obtenido:</strong> {rf_recall:.3f}</p>
    <p>{final_conclusion}</p>
</section>
"""

limitations_html = '<section><h2>Limitaciones</h2><ul>' + ''.join(f'<li>{item}</li>' for item in limitations) + '</ul></section>'
future_work_html = '<section><h2>Trabajos futuros</h2><ul>' + ''.join(f'<li>{item}</li>' for item in future_work) + '</ul></section>'

story_html = f"""
<section><h2>Contexto</h2><p>El dashboard resume el flujo final del TP: datos procesados, hallazgos descriptivos y desempeno predictivo.</p></section>
<section><h2>Hipotesis</h2><p>{cross_validation.get('hypothesis', model_metrics.get('hypothesis', ''))}</p></section>
{imbalance_html}
{model_table_html}
{confusion_matrix_html}
{confusion_interpretation_html}
{error_cost_html}
{conclusions_html}
{limitations_html}
{future_work_html}
"""
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

html = f"""
<!doctype html>
<html lang='es'>
<head>
    <meta charset='utf-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1'>
    <title>Dashboard final - Siniestros viales</title>
    <style>
        :root {{
            --ink: #17202a;
            --muted: #5c6773;
            --line: #d8dee6;
            --panel: #f7f9fb;
            --accent: #2e86ab;
            --winner: #e8f4fb;
        }}
        body {{ margin: 0; font-family: Arial, Helvetica, sans-serif; color: var(--ink); background: #ffffff; }}
        header {{ padding: 32px 48px 22px; border-bottom: 1px solid var(--line); background: var(--panel); }}
        main {{ max-width: 1180px; margin: 0 auto; padding: 28px 28px 48px; }}
        h1 {{ margin: 0 0 8px; font-size: 32px; line-height: 1.15; }}
        h2 {{ margin: 26px 0 8px; font-size: 22px; }}
        p, li {{ color: var(--muted); line-height: 1.55; max-width: 980px; }}
        .kpi-grid {{ display: grid; grid-template-columns: repeat(5, minmax(150px, 1fr)); gap: 12px; margin: 22px 0 16px; }}
        .kpi-card {{ border: 1px solid var(--line); border-radius: 8px; padding: 16px; background: #fff; }}
        .kpi-label {{ color: var(--muted); font-size: 13px; min-height: 34px; }}
        .kpi-value {{ margin-top: 10px; font-size: 24px; font-weight: 700; color: var(--accent); overflow-wrap: anywhere; }}
        .chart-section {{ margin-top: 28px; }}
        .plotly-graph-div {{ margin: 18px 0 12px; border-top: 1px solid var(--line); padding-top: 14px; }}
        .interpretation-block {{ background: #f3f5f7; border: 1px solid var(--line); border-radius: 8px; padding: 14px 16px; margin: 12px 0 28px; max-width: 980px; }}
        .interpretation-block h3 {{ margin: 0 0 6px; font-size: 16px; color: var(--ink); }}
        .interpretation-block p {{ margin: 0; color: var(--muted); }}
        .confusion-matrix {{ max-width: 760px; width: 100%; height: auto; display: block; margin: 18px 0; border: 1px solid var(--line); }}
        .metrics-table {{ border-collapse: collapse; margin: 14px 0 22px; min-width: 420px; }}
        .metrics-table th, .metrics-table td {{ border: 1px solid var(--line); padding: 10px 14px; text-align: right; }}
        .metrics-table th {{ background: var(--panel); }}
        .model-summary td:first-child, .model-summary th:first-child {{ text-align: left; }}
        .winner-row {{ background: var(--winner); font-weight: 700; }}
        footer {{ margin-top: 24px; padding-top: 18px; border-top: 1px solid var(--line); color: var(--muted); font-size: 13px; }}
        @media (max-width: 900px) {{ header {{ padding: 24px; }} .kpi-grid {{ grid-template-columns: repeat(2, minmax(140px, 1fr)); }} .metrics-table {{ min-width: 0; width: 100%; }} }}
    </style>
</head>
<body>
    <header>
        <h1>Dashboard final: siniestros viales</h1>
        <p>Analisis descriptivo, comparacion de modelos y narrativa final para la defensa del TP.</p>
    </header>
    <main>
        <div class='kpi-grid'>{kpi_cards}</div>
        {story_html}
        {plot_html}
        <footer>Generado el {created_at}. Fuente: dataset procesado y artefactos guardados en outputs/.</footer>
    </main>
</body>
</html>
"""

dashboard_path.write_text(html, encoding='utf-8')
logger.info(f'Dashboard exportado en: {dashboard_path}')
print('Dashboard generado')
print(f'Ubicacion: {dashboard_path}')
print(f'Fecha de generacion: {created_at}')
